**Course:** DS200.Q21 - Big Data Analysis (Lab 01)  
**Student:** Nguyen Cong Phat  
**Student ID:** 23521143
 
---

**Dữ liệu:**
1. movies.txt

- **Schema**: MovieID, Title, Genres

2. ratings_1.txt, ratings_2.txt

- **Schema**: UserID, MovieID, Rating, Timestamp

3. users.txt

- **Schema**: UserID, Gender, Age, Occupation, Zip-code


### Bài 1: Tính Điểm Đánh Giá Trung Bình và Tổng Số Lượt Đánh Giá Cho Mỗi Phim

Mục tiêu:
- Tính điểm trung bình cho từng phim từ cả 2 file ratings (ratings_1.txt và ratings_2.txt).
- Tính tổng số lượt đánh giá cho mỗi phim.
- Output: MovieTitle AverageRating: xx (TotalRatings: xx).

Lưu ý:
- Trong quá trình xử lý, tìm ra phim có điểm trung bình cao nhất (chỉ xét những phim có tối thiểu 5 lượt đánh giá).
- Sử dụng biến lớp như maxMovie và maxRating trong reducer và xuất ra kết quả cuối cùng trong phương thức cleanup(), theo định dạng:
MovieTitle is the highest rated movie with an average rating of AverageRating among movies with at least 5 ratings.

![image.png](attachment:image.png)


### Bài 2: Phân Tích Đánh Giá Theo Thể Loại

Mục tiêu:
- Vì một phim có thể thuộc nhiều thể loại (Genres được phân tách bằng dấu “|”), mapper cần tách riêng từng thể loại của phim đó.
- Tính điểm trung bình (và tổng số lượt đánh giá nếu cần) cho từng thể loại, dựa trên tất cả các phim thuộc thể loại đó.

Output: Genre: AverageRating (TotalRatings).

![image.png](attachment:image.png)

### Bài 3: Phân Tích Đánh Giá Theo Giới Tính

Mục tiêu:
- Thực hiện join dữ liệu giữa ratings và users (dựa trên UserID) để lấy thông tin giới tính của người đánh giá.
- Với mỗi phim, tính riêng điểm trung bình từ người dùng nam và nữ.

Output: MovieTitle: Male_Avg, Female_Avg

![image.png](attachment:image.png)

### Bài 4: Phân Tích Đánh Giá Theo Nhóm Tuổi

Mục tiêu:
- Phân nhóm người dùng theo độ tuổi: 0-18, 18-35, 35-50, 50+.
- Với mỗi phim, tính điểm trung bình cho mỗi nhóm tuổi.

Output: MovieTitle: [0-18: AvgRating, 18-35: AvgRating, 35-50: AvgRating, 50+: AvgRating]

![image.png](attachment:image.png)

### Hands-on code (Tasks 1-4, pandas reference)

Run the cells below after `pip install -r requirements.txt`. Reports are written to `output/` when you run the last cell or `scripts/run_all_assignments.py`.


In [ ]:
# Lab setup: find repository root, add src/ to import path, preview raw tables
from pathlib import Path
import sys


def find_project_root() -> Path:
    """Walk upward from cwd until data/movies.txt exists (treat that folder as repo root)."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "movies.txt").exists():
            return p
    raise FileNotFoundError(
        "Project root not found (expected data/movies.txt). "
        "Run Jupyter from the repo root or from notebooks/."
    )


ROOT = find_project_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from lab01.io import load_movies, load_ratings, load_users
from lab01.analytics import (
    task1_average_ratings_per_movie,
    task2_average_ratings_per_genre,
    task3_ratings_by_gender_per_movie,
    task4_ratings_by_age_group_per_movie,
)

DATA_DIR = ROOT / "data"
movies = load_movies(DATA_DIR)
ratings = load_ratings(DATA_DIR)
users = load_users(DATA_DIR)
print(movies.head(), ratings.head(), users.head(), sep="\n\n")


In [ ]:
# Task 1: average rating and total count per movie; summary line for top movie (>= 5 ratings)
text1, df1 = task1_average_ratings_per_movie(DATA_DIR)
print(text1)


In [ ]:
# Task 2: split multi-genre movies (|) and aggregate mean rating per genre
text2, df2 = task2_average_ratings_per_genre(DATA_DIR)
print(text2)


In [ ]:
# Task 3: join ratings with users; male vs female mean rating per movie title
text3, df3 = task3_ratings_by_gender_per_movie(DATA_DIR)
print(text3)


In [ ]:
# Task 4: bucket user ages, then mean rating per movie for each age group
text4, df4 = task4_ratings_by_age_group_per_movie(DATA_DIR)
print(text4)


In [ ]:
# Write the same four reports as scripts/run_all_assignments.py (for submission)
OUT = ROOT / "output"
OUT.mkdir(parents=True, exist_ok=True)
files = [
    ("task1_movie_ratings.txt", text1),
    ("task2_genre_ratings.txt", text2),
    ("task3_gender_by_movie.txt", text3),
    ("task4_age_groups_by_movie.txt", text4),
]
for name, content in files:
    (OUT / name).write_text(content + "\n", encoding="utf-8")
    print("Wrote", OUT / name)


### Hadoop MapReduce (course slides)

Instructor PDFs are in **`../slides/`** (for example *Slide 2 GFS and Hadoop.pdf* and *Slide 3 Hadoop MapReduce Tutorial.pdf*).

The **MapReduce / Hadoop Streaming** solution for Tasks 1–4 lives in **`hadoop/streaming/`** (Python mapper and reducer scripts; English comments throughout).

**Run locally** (the shell script uses `sort` as the shuffle phase; no Hadoop daemons required):

```bash
bash scripts/run_hadoop_streaming_local.sh
```

This regenerates the same four files under **`output/`** as the pandas workflow. On a cluster, package the same scripts with `hadoop jar … hadoop-streaming-*.jar`, `-files`, `-mapper`, and `-reducer` as shown in the slides (`hadoop/run_hadoop_cluster_example.sh` is a starting template).